# Multichannel transcription of Zoom meetings

This notebook shows how to synthesize individual participant recordings from Zoom recordings into one file, and then transcribe each individually.

Learn more in the blog post: [How to transcribe Zoom participant recordings (multichannel)](https://www.assemblyai.com/blog/transcribe-multichannel-zoom/)

## Imports and setup

First are our imports, where we're importing our local `utils` module and `ZoomClient`. We then load environment variables and assign them to the appropriate variables, and ensure that they are all present.

Then we instantiate a `ZoomClient`, which is used to interact with the Zoom API, and an AssemblyAI `Transcriber` object, which is used to transcribe files using AssemblyAI. We pass a `config` into this Transcriber to enable multichannel transcription.

In [1]:
import os

from dotenv import load_dotenv
import assemblyai as aai

import utils
from utils.zoom import ZoomClient

# load environment variables
load_dotenv()

# assign to variables
ZOOM_ACCOUNT_ID = os.environ.get('ZOOM_ACCOUNT_ID')
ZOOM_CLIENT_ID = os.environ.get('ZOOM_CLIENT_ID')
ZOOM_CLIENT_SECRET = os.environ.get('ZOOM_CLIENT_SECRET')
aai.settings.api_key = os.environ.get('ASSEMBLYAI_API_KEY')

# ensure all required environment variables are available
if not all([ZOOM_ACCOUNT_ID, ZOOM_CLIENT_ID, ZOOM_CLIENT_SECRET, aai.settings.api_key]):
    raise EnvironmentError(
        "Missing one or more required environment variables: "
        "ZOOM_ACCOUNT_ID, ZOOM_CLIENT_ID, ZOOM_CLIENT_SECRET, ASSEMBLYAI_API_KEY"
    )

# instantiate Zoom client to interact with Zoom API
client = ZoomClient(account_id=ZOOM_ACCOUNT_ID, client_id=ZOOM_CLIENT_ID, client_secret=ZOOM_CLIENT_SECRET)

# instantiate AssemblyAI transcriber with multichannel speech-to-text enabled
config = aai.TranscriptionConfig(multichannel=True)
transcriber = aai.Transcriber(config=config)

## Cloud Recordings

Next is our main function. We use the `ZoomClient` to get the list of cloud recordings, and then download the individual participant files from the first one.

We then use the `utils.combine_tracks` helper function to combine all of these files into one, with each original file on it's own mono audio channel.

Then we send the file to AssemblyAI for transcription, and print off the number of channels in the original file and the utterances, each of which specifies which channel on which the utterance was spoken.

In [ ]:
def cloud():
    # download each participant audio files for the most recent Zoom meeting
    params = {'from': '2024-11-14'}  # query parameters for Zoom API request
    meets = client.get_recordings(params=params)
    meeting_uuid = meets["meetings"][0]["uuid"]
    client.download_participant_audio_files(meeting_uuid)
    
    # combine all participant audio files into a single audio file
    path = "combined_audio.m4a"
    utils.combine_tracks(path, dir="tmp")
    
    # send to AssemblyAI for multichannel speech-to-text and print the results
    transcript = transcriber.transcribe(path)
    print(transcript.json_response["audio_channels"])
    for utt in transcript.utterances:
        print(utt)

cloud()

2
text='Here I am talking on channel one.' start=3560 end=5105 confidence=0.9430786 speaker='2' channel='2' words=[UtteranceWord(text='Here', start=3560, end=3696, confidence=0.99461, speaker='2', channel='2'), UtteranceWord(text='I', start=3696, end=3840, confidence=0.99855, speaker='2', channel='2'), UtteranceWord(text='am', start=3840, end=4096, confidence=0.99833, speaker='2', channel='2'), UtteranceWord(text='talking', start=4096, end=4400, confidence=0.99964, speaker='2', channel='2'), UtteranceWord(text='on', start=4425, end=4625, confidence=0.9976, speaker='2', channel='2'), UtteranceWord(text='channel', start=4665, end=4969, confidence=0.62153, speaker='2', channel='2'), UtteranceWord(text='one.', start=5017, end=5105, confidence=0.99129, speaker='2', channel='2')]
text='And here I am as a different participant, talking on channel two.' start=8880 end=12025 confidence=0.95788336 speaker='1' channel='1' words=[UtteranceWord(text='And', start=8880, end=9040, confidence=0.93976, 

# Local recordings

For local recordings, place the participant recordings in one directory (`recordings` here) and then combine/transcribe the tracks:

In [ ]:
def local_files():
    # combine all participant audio files into a single audio file
    path = "combined_audio_local.m4a"
    utils.combine_tracks(path, dir="recordings")
    
    # send to AssemblyAI for multichannel speech-to-text and print the results
    transcript = transcriber.transcribe(path)
    print(transcript.json_response["audio_channels"])
    for utt in transcript.utterances:
        print(utt)

local_files()